# IA interna · versión 1 — detector de estructuras (frambuesa)

Entrena el detector de **yemas, brotes, laterales, botones, flores y frutos** con las marcas de IA+
(`AeonCrop/PhenoRubus/Entrenamiento IA/marcas.json`) y las fotos de tu Drive.

**Cómo usarlo**
1. Menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.
2. *Entorno de ejecución → Ejecutar todo*. Cuando lo pida, autoriza el acceso a tu Drive.
3. Tarda ~20–40 min. Al final el modelo queda en
   `AeonCrop/PhenoRubus/Entrenamiento IA/modelos/v1/` (PyTorch, ONNX para la web y TFLite para el teléfono),
   con sus métricas y fotos de ejemplo.

Solo se usan las fotos **marcadas o confirmadas por una persona** (las cajas de Gemini sin revisar quedan fuera).

In [ ]:
# 1) Librerías y Drive
!pip -q install -U ultralytics
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Ajustes (puedes cambiarlos)
ROOT_DRIVE = '/content/drive/MyDrive/AeonCrop'          # carpeta de la app en tu Drive
TRAIN_DIR  = f'{ROOT_DRIVE}/PhenoRubus/Entrenamiento IA'  # aquí está marcas.json
VERSION    = 'v1'
MODEL      = 'yolov8n.pt'   # pequeño y rápido: corre en el teléfono y en el navegador
IMGSZ      = 640            # súbelo a 960 si los botones chicos se detectan mal (más lento)
EPOCHS     = 120
VAL_FRAC   = 0.15
SEED       = 7
# Clases en el mismo orden que la app (training_labels.CLASSES): el índice es el id del modelo.
CLASSES = ['yema', 'brote', 'lateral', 'boton', 'boton_blanco', 'flor', 'flor_pasada',
           'fruto_verde', 'fruto_pinton', 'fruto_maduro']

In [ ]:
# 3) Marcas confirmadas por una persona
import json, os, random, collections
marcas = json.load(open(f'{TRAIN_DIR}/marcas.json'))
fotos = {k: p for k, p in marcas['photos'].items()
         if p.get('by') != 'IA' and not p.get('review') and (p.get('boxes') or p.get('empty'))}
cnt = collections.Counter(b['c'] for p in fotos.values() for b in p.get('boxes', []))
print(len(fotos), 'fotos ·', sum(cnt.values()), 'cajas')
for c in CLASSES: print(f'  {c:14s} {cnt.get(c, 0)}')

In [ ]:
# 4) Buscar cada foto en el Drive (por nombre de archivo)
want = {os.path.basename(k): k for k in fotos}
found = {}
for base, dirs, files in os.walk(ROOT_DRIVE):
    if '/modelos' in base or '/.' in base:
        continue
    for f in files:
        if f in want and want[f] not in found:
            found[want[f]] = os.path.join(base, f)
faltan = [k for k in fotos if k not in found]
print(len(found), 'encontradas ·', len(faltan), 'no están en Drive')
if faltan: print('  ej.:', faltan[:5])

In [ ]:
# 5) Dataset YOLO: fotos derechas (EXIF), máx. 1280 px, y cajas en formato YOLO
from PIL import Image, ImageOps
DS = '/content/ds'
for s in ('train', 'val'):
    os.makedirs(f'{DS}/images/{s}', exist_ok=True); os.makedirs(f'{DS}/labels/{s}', exist_ok=True)
keys = sorted(found); random.Random(SEED).shuffle(keys)
nval = max(1, int(len(keys) * VAL_FRAC))
split = {k: ('val' if i < nval else 'train') for i, k in enumerate(keys)}
skip = 0
for k in keys:
    p = fotos[k]
    try:
        im = ImageOps.exif_transpose(Image.open(found[k])).convert('RGB')
    except Exception as e:
        skip += 1; continue
    W, H = im.size
    # Las marcas se hicieron sobre la foto derecha: si la proporción no calza, no la usamos.
    if p.get('w') and p.get('h') and (p['w'] > p['h']) != (W > H) and abs(W - H) > 8:
        skip += 1; continue
    im.thumbnail((1280, 1280))
    name = k.replace('/', '__').rsplit('.', 1)[0]
    im.save(f'{DS}/images/{split[k]}/{name}.jpg', quality=92)
    with open(f'{DS}/labels/{split[k]}/{name}.txt', 'w') as f:
        for b in p.get('boxes', []):
            if b['c'] not in CLASSES: continue
            x, y, w, h = (min(max(float(b[q]), 0), 1) for q in ('x', 'y', 'w', 'h'))
            if w <= 0 or h <= 0: continue
            f.write(f"{CLASSES.index(b['c'])} {x + w / 2:.6f} {y + h / 2:.6f} {w:.6f} {h:.6f}\n")
yaml = f'{DS}/data.yaml'
open(yaml, 'w').write(f"path: {DS}\ntrain: images/train\nval: images/val\nnames:\n" +
                      ''.join(f'  {i}: {c}\n' for i, c in enumerate(CLASSES)))
print('train', len(os.listdir(f'{DS}/images/train')), '· val', len(os.listdir(f'{DS}/images/val')), '· omitidas', skip)

In [ ]:
# 6) Vista de control: 6 fotos con sus cajas (revisa que calcen antes de entrenar)
import matplotlib.pyplot as plt, matplotlib.patches as pa, glob
fs = sorted(glob.glob(f'{DS}/images/train/*.jpg'))[:6]
fig, ax = plt.subplots(2, 3, figsize=(15, 9))
for a, f in zip(ax.flat, fs):
    im = Image.open(f); a.imshow(im); a.axis('off'); W, H = im.size
    for line in open(f.replace('/images/', '/labels/').replace('.jpg', '.txt')):
        c, x, y, w, h = map(float, line.split())
        a.add_patch(pa.Rectangle(((x - w/2) * W, (y - h/2) * H), w * W, h * H, fill=False, lw=1.5, ec='yellow'))
        a.text((x - w/2) * W, (y - h/2) * H, CLASSES[int(c)], color='yellow', fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# 7) Entrenar
from ultralytics import YOLO
model = YOLO(MODEL)
r = model.train(data=yaml, imgsz=IMGSZ, epochs=EPOCHS, patience=30, batch=16, seed=SEED,
                project='/content/runs', name=VERSION, exist_ok=True,
                fliplr=0.5, flipud=0.0, degrees=10, scale=0.4, mosaic=1.0, close_mosaic=15)

In [ ]:
# 8) Métricas por clase (validación)
best = YOLO(f'/content/runs/{VERSION}/weights/best.pt')
m = best.val(data=yaml, imgsz=IMGSZ, split='val')
print(f'mAP50 = {m.box.map50:.3f} · mAP50-95 = {m.box.map:.3f}')
per = {}
for i, c in enumerate(m.box.ap_class_index):
    per[CLASSES[int(c)]] = {'ap50': float(m.box.ap50[i]), 'p': float(m.box.p[i]), 'r': float(m.box.r[i])}
    print(f"  {CLASSES[int(c)]:14s} AP50 {m.box.ap50[i]:.2f}  precisión {m.box.p[i]:.2f}  recall {m.box.r[i]:.2f}")

In [ ]:
# 9) Exportar: ONNX (web) y TFLite (teléfono)
onnx = best.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
try:
    tfl = best.export(format='tflite', imgsz=IMGSZ)       # float32 (+ tarda unos minutos)
except Exception as e:
    tfl = None; print('TFLite no se pudo exportar:', e)

In [ ]:
# 10) Guardar en Drive: modelos/v1
import shutil, datetime, glob
OUT = f'{TRAIN_DIR}/modelos/{VERSION}'
os.makedirs(OUT, exist_ok=True)
shutil.copy(f'/content/runs/{VERSION}/weights/best.pt', f'{OUT}/detector_{VERSION}.pt')
shutil.copy(onnx, f'{OUT}/detector_{VERSION}.onnx')
if tfl:
    shutil.copy(tfl, f'{OUT}/detector_{VERSION}.tflite')
for f in ('results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg', 'val_batch0_labels.jpg'):
    if os.path.exists(f'/content/runs/{VERSION}/{f}'):
        shutil.copy(f'/content/runs/{VERSION}/{f}', f'{OUT}/{f}')
meta = {'version': VERSION, 'model': MODEL, 'imgsz': IMGSZ, 'classes': CLASSES,
        'trained_at': datetime.datetime.now().isoformat(timespec='seconds'),
        'photos': {'train': len(os.listdir(f'{DS}/images/train')), 'val': len(os.listdir(f'{DS}/images/val'))},
        'boxes': dict(cnt), 'map50': float(m.box.map50), 'map50_95': float(m.box.map), 'per_class': per,
        'input': 'RGB 0-1, letterbox a imgsz×imgsz', 'output': 'YOLOv8: [1, 4+nc, N] (cx, cy, w, h en píxeles del imgsz, luego puntaje por clase)'}
json.dump(meta, open(f'{OUT}/model.json', 'w'), indent=2, ensure_ascii=False)
print('Listo:', OUT); print(os.listdir(OUT))

In [ ]:
# 11) Prueba rápida sobre 4 fotos de validación
res = best.predict(sorted(glob.glob(f'{DS}/images/val/*.jpg'))[:4], imgsz=IMGSZ, conf=0.25)
fig, ax = plt.subplots(1, 4, figsize=(20, 6))
for a, rr in zip(ax, res):
    a.imshow(rr.plot()[..., ::-1]); a.axis('off')
plt.tight_layout(); plt.show()